# Orthogonality Loss Under Perturbation

Chapter 4: QR Decomposition and Numerical Rank · *Mathematics for AI and Machine Learning* by Fuheng Wu

See how nearly collinear columns challenge numerical orthogonalization.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-4/orthogonality-loss/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's setup: a1 = [1, 1] and a nearly parallel a2 = [1+eps+noise, 1+eps-noise]
eps, noise = 0.04, 0.05


def classical_gs(A):
    Q = torch.zeros_like(A)
    for j in range(A.shape[1]):
        v = A[:, j].clone()
        for i in range(j):
            v -= (Q[:, i] @ A[:, j]) * Q[:, i]   # projects the ORIGINAL column
        Q[:, j] = v / v.norm()
    return Q


def modified_gs(A):
    Q = A.clone()
    for j in range(A.shape[1]):
        for i in range(j):
            Q[:, j] -= (Q[:, i] @ Q[:, j]) * Q[:, i]  # projects the UPDATED column
        Q[:, j] /= Q[:, j].norm()
    return Q


def orth_error(Q):
    I = torch.eye(Q.shape[1], dtype=Q.dtype)
    return torch.linalg.matrix_norm(Q.T @ Q - I).item()   # ||Q^T Q - I||_F


A = torch.tensor([[1.0, 1 + eps + noise],
                  [1.0, 1 + eps - noise]], dtype=torch.float32)
print("float32, 2 columns: ||Q^T Q - I||_F =", orth_error(classical_gs(A)))

# ── The loss grows as columns become more parallel (here in float32) ────────
for gap in [1e-1, 1e-2, 1e-3, 1e-4]:
    A = torch.tensor([[1.0, 1.0, 1.0],
                      [gap, 0.0, 0.0],
                      [0.0, gap, 0.0],
                      [0.0, 0.0, gap]], dtype=torch.float32)
    cond = torch.linalg.cond(A.double()).item()
    print(f"gap={gap:.0e}  cond={cond:9.2e}  classical={orth_error(classical_gs(A)):.2e}"
          f"  modified={orth_error(modified_gs(A)):.2e}"
          f"  householder={orth_error(torch.linalg.qr(A).Q):.2e}")
# Classical GS loses orthogonality fastest; Householder QR stays near machine precision.

float32, 2 columns: ||Q^T Q - I||_F = 1.686927248556458e-06
gap=1e-01  cond= 1.73e+01  classical=9.58e-06  modified=1.35e-06  householder=3.40e-07
gap=1e-02  cond= 1.73e+02  classical=7.56e-04  modified=1.07e-05  householder=3.95e-07
gap=1e-03  cond= 1.73e+03  classical=3.78e-02  modified=5.35e-05  householder=2.80e-07
gap=1e-04  cond= 1.73e+04  classical=7.07e-01  modified=1.15e-04  householder=1.79e-07
